# IBM QAOA Data Analysis with Stochastic Benchmark

This notebook processes IBM QAOA experimental data and runs it through the stochastic benchmark framework. We'll convert JSON results into the required format and perform comprehensive analysis.

## 1. Import Required Libraries

Import necessary libraries including json, os, subprocess, and pathlib for file operations and script execution.

In [1]:
import json
import os
import glob
import sys
import re
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from typing import Dict, List, Any
from dataclasses import dataclass
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from typing import List, Dict, Any, Union
from dataclasses import dataclass, field


# Set up matplotlib style
plt.style.use('../../src/ws.mplstyle')

# Add stochastic-benchmark src to path
sys.path.append('../../src')

# Import stochastic benchmark modules
import stochastic_benchmark
import bootstrap
import success_metrics
import interpolate
from interpolate import Interpolate
import stats
import training

print("Libraries imported successfully!")

%load_ext autoreload
%autoreload 2

Libraries imported successfully!


## 2. Load and Parse JSON Configuration

Read and parse the JSON configuration file to extract parameters, file paths, and execution settings.

Read and parse the JSON configuration file from hardware runs to extract parameters, file paths, and execution settings.

In [2]:
# Load the JSON data
cwd = os.getcwd()
print(f"Current working directory: {cwd}")
# Update this path to match your folder structure
json_file_path = os.path.join(cwd, "R3R/Hardware/000N40*.json") 

# Use glob to get a list of all .json file paths
json_files = glob.glob(json_file_path)
print(f"Found files: {json_files}")

all_qaoa_data = []

for json_file in json_files:
    print(f"\nProcessing file: {os.path.basename(json_file)}")
    with open(json_file, 'r') as f:
        data = json.load(f)
        all_qaoa_data.append(data)
    
    # Check if the loaded data is a LIST (New Format)
    if isinstance(data, list):
        print(f"Detected NEW format (List of {len(data)} jobs)")
        
        for i, job in enumerate(data):
            # Use 'job_id' if available, otherwise use index
            job_id = job.get('job_id', str(i))
            print(f"\n--- Job/Trial {job_id} ---")
            
            # In the new format, most info is inside 'metadata'
            metadata = job.get('metadata', {})
            
            # 1. Handle Trainer
            # In new format, 'trainer' is often a simple string in metadata
            if 'trainer' in metadata:
                print(f"Trainer: {metadata['trainer']}")
            
            # 2. Print other metadata (params, method, energy, etc.)
            for key, value in metadata.items():
                if key == 'trainer': 
                    continue # Already printed
                
                # Truncate long lists for cleaner output
                if isinstance(value, list) and len(value) > 3:
                    print(f"{key}: [{value[0]}, {value[1]}, ..., {value[-1]}] (length: {len(value)})")
                else:
                    print(f"{key}: {value}")
            
            # 3. Print Hardware Counts info (unique to hardware files)
            if 'counts' in job:
                num_unique_bitstrings = len(job['counts'])
                print(f"Counts: {num_unique_bitstrings} unique bitstrings measured")

        else:
            print("Warning: Dictionary found, but no numbered trial keys detected.")
    
    else:
        print("Error: Unknown JSON structure.")

Current working directory: /mnt/c/Users/rames102/Desktop/stochastic-benchmark/examples/IBM_QAOA
Found files: ['/mnt/c/Users/rames102/Desktop/stochastic-benchmark/examples/IBM_QAOA/R3R/Hardware/000N40R3R_10_d494bmfnmdfs73abgatg.json', '/mnt/c/Users/rames102/Desktop/stochastic-benchmark/examples/IBM_QAOA/R3R/Hardware/000N40R3R_2_d498d5hlag1s73biv770.json', '/mnt/c/Users/rames102/Desktop/stochastic-benchmark/examples/IBM_QAOA/R3R/Hardware/000N40R3R_4_d498bdvnmdfs73abk9s0.json', '/mnt/c/Users/rames102/Desktop/stochastic-benchmark/examples/IBM_QAOA/R3R/Hardware/000N40R3R_6_d4989aa489vs73a15e90.json']

Processing file: 000N40R3R_10_d494bmfnmdfs73abgatg.json
Detected NEW format (List of 8 jobs)

--- Job/Trial d494bmfnmdfs73abgatg ---
circuit_metadata: {'file_name': 'instances/random_regular/000_40nodes_random3regular.json', 'problem_class': 'maxcut', 'layout_info': {'path': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 19, 35, 34, 33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 3

## 3. File Processing Functions

Create utility functions to process and validate input files, handle different file formats, and prepare data for script execution.

In [3]:
# --- 1. DATA STRUCTURES & PARSERS ---

@dataclass
class QAOAResult:
    """Data class to hold QAOA optimization results"""
    trial_id: str
    optimized_params: List[float]
    train_duration: float
    energy: float
    approximation_ratio: float
    trainer_name: str
    method: str
    success: bool
    energy_history: List[float] = field(default_factory=list)
    counts: Dict[str, int] = field(default_factory=dict)
    all_bitstrings: List[str] = field(default_factory=list)

def expand_counts(counts: Dict[str, int]) -> List[str]:
    """Helper to expand {'00': 2, '01': 1} into ['00', '00', '01']"""
    expanded = []
    if counts:
        for bitstring, count in counts.items():
            expanded.extend([bitstring] * count)
    return expanded

def parse_new_qaoa_job(job: Dict[str, Any], index: int) -> QAOAResult:
    """
    Parser for the NEW list-based JSON format.
    Fix: Looks inside 'circuit_metadata' for energy/params if not found in 'metadata'.
    """
    base_job_id = job.get('job_id', 'unknown')
    # Create a unique ID for each item in the list: "jobID_index"
    unique_trial_id = f"{base_job_id}_{index}"
    
    metadata = job.get('metadata', {})
    
    # Check where the info is hiding. 
    # In your file, it's inside metadata['circuit_metadata']
    if 'circuit_metadata' in metadata and 'eval_energy' in metadata['circuit_metadata']:
        info_source = metadata['circuit_metadata']
    else:
        info_source = metadata # Fallback to standard location

    # Extract Values
    energy = info_source.get('eval_energy', np.nan)
    params = info_source.get('params', [])
    method = info_source.get('method', None)
    trainer_raw = info_source.get('trainer', 'Unknown')
    
    # Extract Counts & Bitstrings
    counts = job.get('counts', {})
    all_bitstrings = expand_counts(counts)
    
    return QAOAResult(
        trial_id=unique_trial_id,
        optimized_params=params,
        train_duration=0.0, 
        energy=energy,
        approximation_ratio=np.nan,
        trainer_name=str(trainer_raw),
        method=method,
        success=True,
        counts=counts,
        all_bitstrings=all_bitstrings
    )

def load_qaoa_results(data: Union[Dict[str, Any], List[Any]]) -> List[QAOAResult]:
    results = []
    if isinstance(data, list):
        for i, job in enumerate(data):
            results.append(parse_new_qaoa_job(job, i))
    elif isinstance(data, dict):
        # (Old dictionary parsing logic if needed)
        pass 
    return results

In [4]:
all_parsed_results = []

for raw_data in all_qaoa_data: # raw_data is the content of one JSON file
    parsed_results = load_qaoa_results(raw_data)
    all_parsed_results.extend(parsed_results)
    
    if parsed_results:
        # Example of accessing the new bitstring data
        print(f"Loaded {len(parsed_results)} trials.")
        first_trial = parsed_results[0]
        print(f"Trial {first_trial.trial_id} has {len(first_trial.all_bitstrings)} total bitstrings.")
        print(first_trial.all_bitstrings[:5]) # Print first 5

Loaded 8 trials.
Trial d494bmfnmdfs73abgatg_0 has 4096 total bitstrings.
['1111110010101110101100010000101100011111', '0001110101011100101000100000110000111111', '0001110000111000001010011110111011101100', '0100010010111000101000100000010010100010', '1111010101110110101000011111000100001100']
Loaded 8 trials.
Trial d498d5hlag1s73biv770_0 has 4096 total bitstrings.
['1111101111111111001100011001110010111101', '1101001100100101101000101111101111010101', '1011111010110000101100001101010111010011', '0100010100100001000100111110110100010111', '1011010000011100011000000111011000001011']
Loaded 8 trials.
Trial d498bdvnmdfs73abk9s0_0 has 4096 total bitstrings.
['1001010011110111110011001111010110001101', '1111000101000011011100010010110111110000', '0100001101010011011101010001110100010100', '0101101111110100110101011101000111110111', '0001010001110110001010101110001010000100']
Loaded 8 trials.
Trial d4989aa489vs73a15e90_0 has 4096 total bitstrings.
['0010101111111110101101100010110101011111', 

## 4. Repository Code Execution Engine

Build functions to discover repository structure, identify executable scripts, and map JSON data to code parameters.

In [5]:
# def convert_to_dataframe(results: List[QAOAResult], instance_id: int = 1, p: int = None) -> pd.DataFrame:
#     """
#     Convert a list of QAOAResult objects into a DataFrame, including bitstring data.
#     """
#     data_rows = []
    
#     for result in results:
#         # Extract parameter values
#         params = result.optimized_params
#         param_dict = {}
#         if len(params) >= 1:
#             param_dict['gamma'] = params[0]
#         if len(params) >= 2:
#             param_dict['beta'] = params[1]
#         for i, param in enumerate(params[2:], start=2):
#             param_dict[f'param_{i}'] = param
            
#         # Assemble row data
#         row = {
#             'trial_id': result.trial_id,
#             'instance': instance_id,
#             'p': p,
#             'Energy': result.energy if not np.isnan(result.energy) else -999,
#             'Approximation_Ratio': result.approximation_ratio if not np.isnan(result.approximation_ratio) else -998,
#             'MeanTime': result.train_duration,
#             'trainer': result.trainer_name,
#             'method': result.method or 'Unknown',
#             'success': result.success,
#             'n_iterations': len(result.energy_history) if result.energy_history else 0,
#             'count': 1,
#             # --- NEW FIELDS FOR BITSTRINGS ---
#             'counts': getattr(result, 'counts', {}),
#             'bitstrings': getattr(result, 'all_bitstrings', []),
#             **param_dict
#         }
#         data_rows.append(row)
    
#     df = pd.DataFrame(data_rows)

#     if df.empty:
#         return df

#     # Approximate GTMinEnergy (ground truth minimum energy)
#     valid_energies = df[df['Energy'] != -999]['Energy']
#     df['GTMinEnergy'] = valid_energies.min() if len(valid_energies) > 0 else -1.0
    
#     # Enforce proper dtypes (only for numeric columns)
#     dtype_map = {
#         'instance': int,
#         'p': int,
#         'gamma': float,
#         'beta': float,
#         'Energy': float,
#         'Approximation_Ratio': float,
#         'MeanTime': float,
#         'count': int
#     }
#     # Only apply astype to columns that actually exist
#     df = df.astype({k: v for k, v in dtype_map.items() if k in df.columns})
    
#     return df


# def prepare_stochastic_benchmark_data(df: pd.DataFrame, instance_id: int, p: int, output_dir: str = 'exp_raw'):
#     """
#     Prepare and save QAOA benchmarking data. Rows with Energy=-999 are removed.
#     """
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
    
#     # Filter out invalid energy rows
#     df_clean = df[df['Energy'] != -999].copy()
    
#     print(f"Original data: {len(df)} rows, Valid data: {len(df_clean)} rows")
    
#     df_final = df_clean.copy()
#     if len(df_final) > 0:
#         df_final['GTMinEnergy'] = df_final['Energy'].min()
    
#     # Save DataFrame to pickle
#     output_file = os.path.join(output_dir, f'raw_results_inst={instance_id}_{p}.pkl')
#     df_final.to_pickle(output_file)
#     print(f"Saved instance {instance_id} → {output_file} ({len(df_clean)} trials)")
    
#     return output_file

# --- 2. DATAFRAME CONVERSION ---

def convert_to_dataframe(results: List[QAOAResult], instance_id: int = 1, p: int = None) -> pd.DataFrame:
    data_rows = []
    
    for result in results:
        params = result.optimized_params
        param_dict = {}
        if len(params) >= 1: param_dict['gamma'] = params[0]
        if len(params) >= 2: param_dict['beta'] = params[1]
        for i, param in enumerate(params[2:], start=2):
            param_dict[f'param_{i}'] = param
            
        row = {
            'trial_id': result.trial_id,
            'instance': instance_id,
            'p': p,
            'Energy': result.energy if not np.isnan(result.energy) else -999,
            'Approximation_Ratio': result.approximation_ratio if not np.isnan(result.approximation_ratio) else -998,
            'MeanTime': result.train_duration,
            'trainer': result.trainer_name,
            'method': result.method or 'Unknown',
            'success': result.success,
            'count': 1,
            'counts': result.counts,
            'bitstrings': result.all_bitstrings,
            **param_dict
        }
        data_rows.append(row)
    
    df = pd.DataFrame(data_rows)
    if df.empty: return df

    # Calculate GTMinEnergy from valid energies
    valid_energies = df[df['Energy'] != -999]['Energy']
    df['GTMinEnergy'] = valid_energies.min() if len(valid_energies) > 0 else -1.0
    
    return df

def prepare_stochastic_benchmark_data(df: pd.DataFrame, instance_id: int, p: int, output_dir: str = 'exp_raw'):
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    
    # Filter out invalid energy rows
    df_clean = df[df['Energy'] != -999].copy()
    
    if not df_clean.empty:
        df_clean['GTMinEnergy'] = df_clean['Energy'].min()
        output_file = os.path.join(output_dir, f'raw_results_inst={instance_id}_{p}.pkl')
        df_clean.to_pickle(output_file)
        print(f"Saved instance {instance_id} (p={p}) -> {output_file}")
        return output_file
    else:
        print(f"Skipping save for instance {instance_id} (p={p}): No valid energy data.")
        return None

In [6]:
# # 1. Parse the data, keeping it grouped by file
# all_qaoa_results = [] 

# # 'all_qaoa_data' comes from your initial file loading step
# for raw_data in all_qaoa_data:
#     # Parse the results for this specific file
#     file_results = load_qaoa_results(raw_data)
    
#     # APPEND (do not extend) to keep results grouped by file
#     all_qaoa_results.append(file_results)

# print(f"Grouped results for {len(all_qaoa_results)} files created.")


# # 2. Run the Dataframe Conversion Loop
# all_qaoa_df = []
# all_data_files = []

# for qaoa_results, json_file in zip(all_qaoa_results, json_files):
    
#     # --- Extract Metadata from Filename ---
#     filename = os.path.basename(json_file)
#     parts = filename.split('_')
    
#     # 1. Extract Instance ID (e.g., from '000N40R3R')
#     instance_str = parts[0]
#     match = re.match(r'(\d+)', instance_str)
#     instance_id = int(match.group(1)) if match else 0
    
#     # 2. Extract p (e.g., '2' from '..._2_...')
#     # New format: '000N40R3R_2_d498...' -> parts[1] is '2'
#     if len(parts) >= 2 and parts[1].isdigit():
#         p = int(parts[1])
#     else:
#         # Fallback for old format
#         try:
#             p = int(parts[-1].replace('.json', ''))
#         except ValueError:
#             p = -1

#     # --- Convert to DataFrame ---
#     qaoa_df = convert_to_dataframe(qaoa_results, instance_id, p)
    
#     if not qaoa_df.empty:
#         print(f"\nInstance {instance_id} (p={p}): DataFrame shape {qaoa_df.shape}")
        
#         all_qaoa_df.append(qaoa_df)

#         # Prepare and save stochastic benchmark data
#         data_file = prepare_stochastic_benchmark_data(qaoa_df, instance_id, p)
#         all_data_files.append(data_file)
#     else:
#         print(f"\nInstance {instance_id}: Empty DataFrame (no results found)")

# print("\nAll instances processed.")

# # 3. Aggregate into a single master DataFrame
# if all_qaoa_df:
#     agg_df = pd.concat(all_qaoa_df, ignore_index=True)
#     agg_df['instance'] = agg_df['instance'].astype(int)
#     agg_df = agg_df.sort_values(by=['instance', 'p']).reset_index(drop=True)
    
#     print("\nAggregated DataFrame:")
#     display(agg_df[['trial_id', 'instance', 'p', 'Energy', 'bitstrings']].head())

cwd = os.getcwd()
# Update this pattern to match your file location
json_files = glob.glob(os.path.join(cwd, "000N40*.json")) 

all_qaoa_results = []
valid_json_files = []

for json_file in json_files:
    with open(json_file, 'r') as f:
        raw_data = json.load(f)
    
    file_results = load_qaoa_results(raw_data)
    if file_results:
        all_qaoa_results.append(file_results)
        valid_json_files.append(json_file)

all_qaoa_df = []

for qaoa_results, json_file in zip(all_qaoa_results, valid_json_files):
    
    filename = os.path.basename(json_file)
    parts = filename.split('_')
    
    # Extract Instance ID
    match = re.match(r'(\d+)', parts[0])
    instance_id = int(match.group(1)) if match else 0
    
    # Extract p value
    if len(parts) >= 2 and parts[1].isdigit():
        p = int(parts[1])
    else:
        try:
            p = int(parts[-1].replace('.json', ''))
        except ValueError:
            p = -1

    qaoa_df = convert_to_dataframe(qaoa_results, instance_id, p)
    
    if not qaoa_df.empty:
        all_qaoa_df.append(qaoa_df)
        prepare_stochastic_benchmark_data(qaoa_df, instance_id, p)

if all_qaoa_df:
    agg_df = pd.concat(all_qaoa_df, ignore_index=True)
    agg_df['instance'] = agg_df['instance'].astype(int)
    agg_df = agg_df.sort_values(by=['instance', 'p']).reset_index(drop=True)
    
    print("\nAggregated DataFrame Head:")
    # Display columns to confirm correction
    display(agg_df[['trial_id', 'instance', 'p', 'Energy', 'method']].head())
else:
    print("No valid data found.")

No valid data found.


In [ ]:
# === Consolidate QAOA results with depth info ===
plot_data = []

for file, qaoa_results in zip(json_files, all_qaoa_results):
    filename = os.path.basename(file)
    try:
        depth = int(filename.split('_')[-1].split('.')[0])
    except (IndexError, ValueError):
        depth = None

    for res in qaoa_results:
        if not np.isnan(res.energy):
            plot_data.append({
                'Approximation Ratio': res.approximation_ratio,
                'Mean Time (s)': res.train_duration,
                'Depth': depth
            })

plot_df = pd.DataFrame(plot_data)

# === Plot ===
if not plot_df.empty:
    plt.figure(figsize=(8, 6))
    depths = sorted(plot_df['Depth'].dropna().unique())

    # Use discrete colors for each depth
    cmap = cm.get_cmap('tab10', len(depths))  # tab10 gives 10 distinct colors
    depth_to_color = {d: cmap(i) for i, d in enumerate(depths)}

    # Scatter points for each depth separately for clear colors and legend
    for d in depths:
        df_d = plot_df[plot_df['Depth'] == d]
        plt.scatter(
            df_d['Mean Time (s)'],
            df_d['Approximation Ratio'],
            color=depth_to_color[d],
            s=80,
            edgecolor='k',
            alpha=0.85,
            label=f'p = {d}'
        )

    # === Formatting ===
    plt.xlabel('Mean Time (s)', fontsize=13)
    plt.ylabel('Approximation Ratio', fontsize=13)
    plt.xscale('log')
    plt.title(
        'Approximation Ratio vs Mean Time\nRandom 3-Regular Graph (N = 10) MaxCut',
        fontsize=15, fontweight='bold', pad=15
    )
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=11, title_fontsize=12)
    plt.tight_layout()
    plt.show()

else:
    print("No data available to plot.")

Create a Stochastic Benchmark Object

In [ ]:
def setup_qaoa_benchmark():
    """
    Initialize and configure the stochastic benchmark object for QAOA experiments.

    This function sets up a `stochastic_benchmark` instance tailored for analyzing
    Quantum Approximate Optimization Algorithm (QAOA) results. It defines parameter names,
    response metrics, and relevant configuration options for memory management and smoothing.
    The setup is designed to streamline benchmarking workflows that involve
    bootstrapping and interpolation across multiple QAOA instances.

    Returns
    -------
    tuple
        A tuple containing:
        - sb : stochastic_benchmark.stochastic_benchmark
            Configured stochastic benchmark object for QAOA analysis.
        - parameter_names : list of str
            List of QAOA variational parameter names used in the analysis 
            (e.g., ['gamma', 'beta', 'p']).

    Notes
    -----
    - The function assumes experimental data is located in the `'exp_raw'` directory.
    - `instance_cols` is explicitly set to `['instance']` to group results by instance ID.
    - By default:
        * `recover=True` enables reuse of stored results.
        * `reduce_mem=True` allows segmented bootstrapping to reduce memory footprint.
        * `smooth=True` enforces monotonicity on the virtual best curve.

    Examples
    --------
    >>> sb, param_names = setup_qaoa_benchmark()
    Stochastic benchmark object created
    Parameter names: ['gamma', 'beta', 'p']
    Instance columns: ['instance']

    See Also
    --------
    setup_bootstrap_parameters : Configures bootstrap iteration and success metrics.
    """
    
    # Define parameter names for QAOA
    parameter_names = ['gamma', 'beta', 'p']
    response_key = 'PerfRatio'
    response_dir = 1

    # Optimization configuration flags
    recover = True       # Whether to reuse existing dataframes when available
    reduce_mem = True    # Whether to segment bootstrapping/interpolation for memory efficiency
    smooth = True        # Whether virtual best should be monotonic (non-decreasing)

    # Create stochastic benchmark instance
    sb = stochastic_benchmark.stochastic_benchmark(
        parameter_names=parameter_names,
        response_key=response_key,
        here='exp_raw',           # Directory containing experiment data
        instance_cols=['instance'],  # Define instance grouping column
        smooth=smooth
    )
    
    print("Stochastic benchmark object created")
    print(f"Parameter names: {parameter_names}")
    print(f"Instance columns: {sb.instance_cols}")
    
    return sb, parameter_names


print("Setting up QAOA stochastic benchmark configuration...")
sb, parameter_names = setup_qaoa_benchmark()


## 5. Script Generation and Validation

Generate executable scripts based on JSON configuration and repository code, with validation and error handling.

Define Bootstrap Parameters

In [ ]:
def setup_bootstrap_parameters():
    """
    Configure and initialize bootstrap parameters for QAOA stochastic benchmarking.

    This function sets up the configuration required for performing stochastic bootstrap
    analysis on Quantum Approximate Optimization Algorithm (QAOA) results. It defines
    shared parameters (e.g., response and resource columns, confidence levels), metric-specific
    arguments (e.g., success probability and runtime tolerance), and links them with an
    update rule function that adapts parameters for each instance based on empirical data.

    Returns
    -------
    bsparams_iter : bootstrap.BSParams_range_iter
        An iterator object that generates bootstrap configurations for different iteration counts.
        This object can be passed to stochastic-benchmark routines for running bootstrap-based
        uncertainty quantification and success metric estimation.

    Notes
    -----
    - The bootstrap configuration is tailored for QAOA performance analysis,
      where the response is typically the *Approximation Ratio* and the resource is the *MeanTime*.
    - The `update_rules()` inner function ensures that each instance's bootstrap run
      uses its specific ground-truth energy (`GTMinEnergy`) and runtime scaling factor.
    - Bootstrap iterations are defined over the range [10, 50] in increments of 10.

    Components
    ----------
    shared_args : dict
        Global parameters for the bootstrap process.
        - 'response_col' : Column name representing the optimization metric (e.g., 'Approximation_Ratio').
        - 'resource_col' : Column name representing computational time (e.g., 'MeanTime').
        - 'response_dir' : Direction of optimization (+1 for maximize, -1 for minimize).
        - 'confidence_level' : Confidence level (in %) for bootstrap intervals.
        - 'random_value' : Seed or offset for reproducibility.

    metric_args : dict
        Metric-specific configurations used to evaluate bootstrap success metrics.
        Includes:
        - 'Response' : Optimization sense.
        - 'SuccessProb' : Success threshold definition.
        - 'RTT' : Runtime-to-target scaling parameters.

    success_metrics : list
        List of success metric functions imported from `success_metrics`, defining
        how performance, success probability, and resource efficiency are evaluated.

    Examples
    --------
    >>> bsparams_iter = setup_bootstrap_parameters()
    Bootstrap configuration:
      Bootstrap iterations: [10, 20, 30, 40, 50]
      Success metrics: ['Response', 'PerfRatio', 'SuccessProb', 'Resource']
      Confidence level: 68%
      Response direction: maximize

    >>> for bs_params in bsparams_iter:
    ...     print(bs_params.shared_args)
    ...     # Used for stochastic bootstrap runs on QAOA datasets
    """
    # Bootstrap parameters
    shared_args = {
        'response_col': 'Approximation_Ratio',
        'resource_col': 'MeanTime',
        'response_dir': 1,  # Maximize approximation ratio
        'confidence_level': 68,
        'random_value': 0.0
    }

    metric_args = {}
    metric_args['Response'] = {'opt_sense': -1}
    metric_args['SuccessProb'] = {'gap': 0.1, 'response_dir': 1}  # Success within 10% of optimum
    metric_args['RTT'] = {
        'fail_value': np.nan,
        'RTT_factor': 1.0,
        'gap': 0.1,
        's': 0.99
    }

    def update_rules(self, df):
        """
        Update bootstrap configuration parameters based on the provided data.

        This method extracts key metrics (e.g., ground truth minimum energy and mean runtime)
        from a given DataFrame and updates internal benchmarking parameters accordingly.
        It ensures that each instance in the bootstrap analysis is properly scaled
        according to its own performance profile.

        Parameters
        ----------
        df : pandas.DataFrame
            Input DataFrame containing QAOA or optimization results for a specific instance.
            Must include columns:
            - 'GTMinEnergy' : float
            - 'MeanTime' : float
        """
        GTMinEnergy = df['GTMinEnergy'].iloc[0]
        self.shared_args['best_value'] = GTMinEnergy
        self.metric_args['RTT']['RTT_factor'] = df['MeanTime'].iloc[0]

    # Success metrics
    sms = [
        success_metrics.Response,
        success_metrics.PerfRatio,
        success_metrics.SuccessProb,
        success_metrics.Resource
    ]

    # Use standard bootstrap range
    boots_range = range(10, 51, 10)  # [10, 20, 30, 40, 50]

    bsParams = bootstrap.BootstrapParameters(
        shared_args=shared_args,
        update_rule=update_rules,
        agg='count',
        metric_args=metric_args,
        success_metrics=sms,
        keep_cols=[]  # Avoid conflicts by not retaining extra columns
    )

    bs_iter_class = bootstrap.BSParams_range_iter()
    bsparams_iter = bs_iter_class(bsParams, boots_range)

    print(f"Bootstrap configuration:")
    print(f"  Bootstrap iterations: {list(boots_range)}")
    print(f"  Success metrics: {[sm.__name__ for sm in sms]}")
    print(f"  Confidence level: {shared_args['confidence_level']}%")
    print(f"  Response direction: {'minimize' if shared_args['response_dir'] == -1 else 'maximize'}")

    return bsparams_iter

# Set up the benchmark configuration
bsparams_iter = setup_bootstrap_parameters()
print("Configuration complete")

## 6. Execute Generated Scripts

Run the generated scripts with proper error handling, logging, and output capture for validation and debugging.

In [ ]:
# Group name function for file parsing
def group_name_fcn(raw_filename):
    """
    Extract the group or instance name from a raw results filename.

    This function assumes that the filename contains the substring 'inst'
    followed by an identifier (e.g., 'inst=001_depth=3.pkl'), and returns
    the portion of the filename from 'inst' up to (but not including) the
    first '.' character.

    Parameters
    ----------
    raw_filename : str
        Full or relative path to the raw results file. The path is reduced
        to its basename before parsing.

    Returns
    -------
    str
        Extracted group or instance identifier string (e.g., 'inst=001_depth=3').

    Raises
    ------
    ValueError
        If the filename does not contain the substring 'inst' or a '.' character.

    Examples
    --------
    >>> group_name_fcn('exp_raw/raw_results_inst=001_2.pkl')
    'inst=001_depth=2'

    >>> group_name_fcn('raw_results_inst=10.pkl')
    'inst=10'
    """
    raw_filename = os.path.basename(raw_filename)
    start_idx = raw_filename.index('inst')
    end_idx = raw_filename.index('.')
    return raw_filename[start_idx:end_idx]

# Debug: Check the data files available
data_files = glob.glob('exp_raw/*.pkl')
print(f"Available data files: {data_files}")
for f in data_files:
    print(f"  - {f}")

if data_files:
    # Load and inspect the first data file
    test_df = pd.read_pickle(data_files[0])
    print(f"\nData file structure:")
    print(f"Shape: {test_df.shape}")
    print(f"Columns: {list(test_df.columns)}")
    print(f"Instance values: {test_df['instance'].unique()}")
    print(f"Sample data:")
    display(test_df.head(3))
else:
    raise FileNotFoundError("No instance data files found in 'exp_raw/' directory.")

# Run Bootstrap for all instances
all_bs_results = []

print("\nRunning bootstrap analysis for all instances...\n")
for file_index, data_file in enumerate(data_files, start=1):
    print(f" Instance {file_index}: {data_file}")
    try:
        raw_data = pd.read_pickle(data_file)
        n_trials = len(raw_data)
        print(f"  → Loaded {n_trials} trials")

        if n_trials == 1:
            print("Single trial detected - using direct analysis without bootstrap aggregation")
            instance_name = group_name_fcn(data_file)

            # Create one DataFrame per bootstrap sample size
            single_instance_results = []
            for n_boots in [10, 20, 30, 40, 50]:
                trial_data = raw_data.iloc[0]
                result_row = {
                    'instance': instance_name,
                    'boots': n_boots,
                    'gamma': trial_data['gamma'],
                    'beta': trial_data['beta'],
                    'p': trial_data['p'],
                    'Key=Response': trial_data['Energy'],
                    'Key=PerfRatio': trial_data['Approximation_Ratio'],
                    'Key=SuccProb': 1.0 if trial_data['success'] else 0.0,
                    'Key=MeanTime': trial_data['MeanTime'],
                    'ConfInt=lower_Key=Response': trial_data['Energy'],
                    'ConfInt=upper_Key=Response': trial_data['Energy'],
                    'ConfInt=lower_Key=PerfRatio': trial_data['Approximation_Ratio'],
                    'ConfInt=upper_Key=PerfRatio': trial_data['Approximation_Ratio'],
                    'ConfInt=lower_Key=SuccProb': 1.0 if trial_data['success'] else 0.0,
                    'ConfInt=upper_Key=SuccProb': 1.0 if trial_data['success'] else 0.0,
                    'ConfInt=lower_Key=MeanTime': trial_data['MeanTime'],
                    'ConfInt=upper_Key=MeanTime': trial_data['MeanTime']
                }
                single_instance_results.append(result_row)

            # Convert all rows for this instance into a DataFrame before appending
            instance_df = pd.DataFrame(single_instance_results)
            all_bs_results.append(instance_df)
            print(f"Bootstrap analysis completed: {instance_df.shape}")

        else:
            print(f"Multiple trials detected ({n_trials}) - attempting standard bootstrap")
            try:
                sb.run_Bootstrap(bsparams_iter, group_name_fcn)
                if sb.bs_results is not None:
                    instance_results = sb.bs_results.copy()
                    instance_results['instance_file'] = data_file
                    all_bs_results.append(instance_results)
                    print(f"Standard bootstrap analysis completed → {instance_results.shape[0]} rows.")
                else:
                    print("Bootstrap returned no results")
            except Exception as e:
                print(f"Standard bootstrap failed for {data_file}: {e}")
                sb.bs_results = None

    except Exception as e:
        print(f"Bootstrap analysis failed: {e}")
        import traceback
        traceback.print_exc()


# Combine bootstrap results
print("\nCombining bootstrap results...")

if len(all_bs_results) > 0:
    combined_bs_results = pd.concat(all_bs_results, ignore_index=True)

    print(f"\n Combined bootstrap results across all instances: {combined_bs_results.shape}")
    print(f"Columns: {list(combined_bs_results.columns)}")

    # Save combined results
    combined_file = 'exp_raw/all_bootstrap_results.pkl'
    combined_bs_results.to_pickle(combined_file)
    print(f"Saved combined bootstrap results → {combined_file}")

    display(combined_bs_results)
else:
    print("\n No bootstrap results were produced across any instance.")


In [ ]:
# Check if we have bootstrap results and handle single instance case for interpolation
if 'combined_bs_results' in locals() and combined_bs_results is not None:
    print("Available columns in bootstrap results:")
    print(list(combined_bs_results.columns))
    print("\nSample bootstrap data:")
    display(combined_bs_results.head(2))

    # Ensure DataFrame
    if isinstance(combined_bs_results, dict):
        combined_bs_results = pd.DataFrame(combined_bs_results)

    n_rows = len(combined_bs_results)
    print(f"\nBootstrap results contain {n_rows} entries")
    
    # Check if this is a single instance case
    if n_rows <= 5:  # Single instance with different bootstrap sizes. Instead should check for number of isntances directly from the instance column.
        print("\nSingle instance detected - skipping interpolation")
        print("For single instance analysis, interpolation provides limited value")
        print("Using bootstrap results directly for further analysis")
        
        # Use bootstrap results as interpolation results
        sb.interp_results = combined_bs_results.copy()
        
        # Add resource column for consistency
        if 'resource' not in sb.interp_results.columns:
            if 'Key=MeanTime' in sb.interp_results.columns:
                sb.interp_results['resource'] = sb.interp_results['Key=MeanTime']
            else:
                sb.interp_results['resource'] = sb.interp_results['boots'] * 0.01  # Synthetic resource based on bootstrap size
        
        print(f"Bootstrap results prepared as interpolation substitute: {sb.interp_results.shape}")
        
    else:
        # Multiple instances - attempt proper interpolation
        print("\nMultiple instances detected - attempting interpolation")

        sb.bs_results = combined_bs_results.copy()
        
        # Set up interpolation parameters with manual resource values
        def resource_fcn(df):
            """Resource function: use appropriate resource column"""
            if 'Key=Resource' in df.columns:
                return df['Key=Resource']
            elif 'Key=MeanTime' in df.columns:
                return df['Key=MeanTime']
            else:
                # Fallback: create artificial resource values
                return pd.Series(np.linspace(0.1, 10.0, len(df)), index=df.index)

        # Create manual resource values
        # simple_resource_values = [0.1, 0.5, 1.0, 2.0, 5.0]
        
        try:
            iParams = interpolate.InterpolationParameters(
                resource_fcn,
                parameters=parameter_names,
                ignore_cols=['trainer', 'method'],
                # resource_value_type='manual',
                # resource_values= 'Key=MeanTime'           # simple_resource_values
            )
            
            print("Running interpolation analysis...")
            # sb.run_Interpolate(iParams)
            sb.interp_results = Interpolate(combined_bs_results, iParams, group_on='instance')
            print("Interpolation analysis completed")
            
            if hasattr(sb, 'interp_results') and sb.interp_results is not None:
                print(f"Interpolation results shape: {sb.interp_results.shape}")
                print("Sample interpolation results:")
                display(sb.interp_results)
            
        except Exception as e:
            print(f"Interpolation failed: {e}")
            print("Using bootstrap results as fallback...")
            sb.interp_results = combined_bs_results.copy()
            
            # Add resource column
            if 'resource' not in sb.interp_results.columns:
                if 'Key=MeanTime' in sb.interp_results.columns:
                    sb.interp_results['resource'] = sb.interp_results['Key=MeanTime']
                else:
                    sb.interp_results['resource'] = np.linspace(0.1, 1.0, len(sb.interp_results))

else:
    print("No bootstrap results available - cannot proceed with interpolation")

In [ ]:
# Check interpolation status and provide alternative if needed
print("\n=== Checking Analysis Status ===")

# Check if interpolation completed successfully
if sb.interp_results.equals(combined_bs_results) is True:
    print(f"Interpolation used Bootstrap results as fallback: {sb.interp_results.shape}")
    print("Sample interpolation results:")
    display(sb.interp_results)

elif hasattr(sb, 'interp_results') and sb.interp_results is not None:
    print(f"Interpolation completed successfully: {sb.interp_results.shape}")
    print("Sample interpolation results:")
    display(sb.interp_results)
else:
    print("No bootstrap results available - analysis cannot continue")
    
# Add train/test split column if missing
if 'train' not in sb.interp_results.columns: # Train is 1, test is 0
    # Simple random train/test split
    np.random.seed(42)
    train_mask = np.random.random(len(sb.interp_results)) < 0.8
    sb.interp_results['train'] = train_mask.astype(int)

print(f"Bootstrap results prepared as interpolation substitute: {sb.interp_results.shape}")
print("Available columns:", list(sb.interp_results.columns))
print("Sample data:")
display(sb.interp_results)

In [ ]:
# Set up and run statistics
print("Running statistics analysis...")
train_test_split = 0.8
# Note: 'Approximation_Ratio' is the response_key, but we use 'PerfRatio' which is automatically created.
metrics = ['Key=PerfRatio', 'Key=MeanTime'] 
stParams = stats.StatsParameters(metrics=metrics, stats_measures=[stats.Median()])

# The check for single instance is already handled inside run_Stats
sb.run_Stats(stParams, train_test_split)

if hasattr(sb, 'stat_results') and sb.stat_results is not None:
    print("Statistics analysis completed.")
    print(f"Statistics results shape: {sb.stat_results.shape}")
    print("Sample statistics results:")
    display(sb.stat_results.head())
else:
    print("Statistics analysis was skipped or failed.")

In [ ]:
# === Simplified QAOA Analysis using Interpolated or Bootstrap Results ===
print("=== Simplified QAOA Analysis ===")

# Use interpolation results if available (includes fallback bootstrap results)
if hasattr(sb, 'interp_results') and sb.interp_results is not None and not sb.interp_results.empty:
    bs_data = sb.interp_results
    print("Analysis source: Interpolation or Bootstrap Fallback Data")
    print(f"Data shape: {bs_data.shape}")
    
    # === Sanity Check on Available Columns ===
    expected_cols = ['Key=PerfRatio', 'Key=Response', 'Key=MeanTime', 'boots']
    missing_cols = [col for col in expected_cols if col not in bs_data.columns]
    if missing_cols:
        print(f"\n Warning: Missing expected columns: {missing_cols}")
    else:
        print("\n All expected columns are available for analysis.")
    
    # === Metric Analysis ===
    metric_columns = [col for col in bs_data.columns if 'Key=' in col or 'ConfInt=' in col]
    if metric_columns:
        print(f"\nAvailable metrics: {metric_columns}")
        print(f"\nMetric Summary:")
        for col in metric_columns:
            if np.issubdtype(bs_data[col].dtype, np.number):
                print(f"{col}:")
                print(f"  Mean: {bs_data[col].mean():.4f}")
                print(f"  Std:  {bs_data[col].std():.4f}")
                print(f"  Min:  {bs_data[col].min():.4f}")
                print(f"  Max:  {bs_data[col].max():.4f}")
    else:
        print("\n No metric columns found for summary statistics.")
    
    # === Parameter Analysis ===
    param_cols = ['gamma', 'beta']
    available_params = [col for col in param_cols if col in bs_data.columns]
    
    if available_params:
        print(f"\nParameter Analysis (from bootstrap/interp data):")
        for param in available_params:
            if np.issubdtype(bs_data[param].dtype, np.number):
                param_range = bs_data[param].max() - bs_data[param].min()
                print(f"{param}: [{bs_data[param].min():.3f}, {bs_data[param].max():.3f}] (range: {param_range:.3f})")
    else:
        # Fallback to original QAOA data if available
        print(f"\nParameter Analysis (from original QAOA data):")
        if 'qaoa_df' in locals() and not qaoa_df.empty:
            for param in ['gamma', 'beta']:
                if param in qaoa_df.columns:
                    param_range = qaoa_df[param].max() - qaoa_df[param].min()
                    print(f"{param}: [{qaoa_df[param].min():.3f}, {qaoa_df[param].max():.3f}] (range: {param_range:.3f})")
    
    # === Instance Check ===
    n_unique_instances = bs_data.get('instance', pd.Series(['inst=1'])).nunique()
    print(f"\nInstance Analysis: {n_unique_instances} unique instance(s)")
    
    # === Performance Stability ===
    if 'Key=PerfRatio' in bs_data.columns:
        perf_std = bs_data['Key=PerfRatio'].std()
        mean_perf = bs_data['Key=PerfRatio'].mean()
        if mean_perf > 0:
            perf_cv = perf_std / mean_perf * 100
            print(f"Performance stability: CV = {perf_cv:.2f}%")
    else:
        print(" Skipping performance stability analysis (no PerfRatio column).")
    
    # === Visualization ===
    print("\nGenerating diagnostic plots...")
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    plotted = False  # Track if at least one plot was drawn
    
    # Performance ratio distribution
    if 'Key=PerfRatio' in bs_data.columns:
        axes[0,0].hist(bs_data['Key=PerfRatio'], bins=min(20, len(bs_data)//2), alpha=0.7, color='steelblue')
        axes[0,0].set_title('Performance Ratio Distribution')
        axes[0,0].set_xlabel('Performance Ratio')
        axes[0,0].set_ylabel('Frequency')
        axes[0,0].grid(True, alpha=0.3)
        mean_perf = bs_data['Key=PerfRatio'].mean()
        axes[0,0].axvline(mean_perf, color='red', linestyle='--', alpha=0.8, label=f'Mean: {mean_perf:.3f}')
        axes[0,0].legend()
        plotted = True
    
    # Response distribution
    if 'Key=Response' in bs_data.columns:
        axes[0,1].hist(bs_data['Key=Response'], bins=min(15, len(bs_data)//2), alpha=0.7, color='steelblue')
        axes[0,1].set_title('Response (Energy) Distribution')
        axes[0,1].set_xlabel('Energy')
        axes[0,1].set_ylabel('Frequency')
        axes[0,1].grid(True, alpha=0.3)
        mean_response = bs_data['Key=Response'].mean()
        axes[0,1].axvline(mean_response, color='red', linestyle='--', alpha=0.8, label=f'Mean: {mean_response:.4f}')
        axes[0,1].legend()
        plotted = True
    
    # Bootstrap sample size effect
    if 'boots' in bs_data.columns and 'Key=PerfRatio' in bs_data.columns:
        axes[1,0].plot(bs_data['boots'], bs_data['Key=PerfRatio'], 'o', alpha=0.7, color='steelblue', markersize=6)
        axes[1,0].set_title('Performance vs Bootstrap Sample Size')
        axes[1,0].set_xlabel('Bootstrap Sample Size')
        axes[1,0].set_ylabel('Performance Ratio')
        axes[1,0].grid(True, alpha=0.3)
        plotted = True
    
    # Time vs Response
    if 'Key=PerfRatio' in bs_data.columns and 'Key=MeanTime' in bs_data.columns:
        axes[1,1].scatter(bs_data['Key=MeanTime'], bs_data['Key=PerfRatio'], alpha=0.7, color='steelblue')
        axes[1,1].set_title('PerfRatio vs Training Time')
        axes[1,1].set_xlabel('Training Time')
        axes[1,1].set_ylabel('Performance Ratio')
        axes[1,1].grid(True, alpha=0.3)
        plotted = True
    
    plt.tight_layout()
    if plotted:
        plt.show()
    else:
        plt.close()
        print(" No plots generated — required columns missing.")
    
    # === Summary ===
    print(f"\n=== QAOA Analysis Summary ===")
    if 'Key=PerfRatio' in bs_data.columns:
        print(f"Mean Performance ratio: {bs_data['Key=PerfRatio'].mean():.4f}")
    if 'Key=Response' in bs_data.columns:
        print(f"Mean Energy: {bs_data['Key=Response'].mean():.4f}")
    if 'qaoa_df' in locals() and not qaoa_df.empty:
        print(f"Original QAOA parameters: gamma={qaoa_df['gamma'].iloc[0]:.3f}, beta={qaoa_df['beta'].iloc[0]:.3f}")
    
    print("\n Analysis completed successfully.")

else:
    print(" No interpolation or bootstrap results available — cannot perform QAOA analysis.")


In [ ]:
# === Check if we should run statistics analysis for single instance ===
if hasattr(sb, 'interp_results') and sb.interp_results is not None and not sb.interp_results.empty:
    n_rows = len(sb.interp_results)
    print(f"Data available for statistics: {n_rows} rows")
    
    # Detect single instance scenario
    n_unique_instances = sb.interp_results.get('instance', pd.Series(['inst=1'])).nunique()
    
    if n_unique_instances == 1 or n_rows <= 5:
        print("\nSingle instance detected - statistics analysis provides limited value")
        print("Skipping statistics analysis for single instance case")
        print("Reason: Insufficient data for meaningful train/test split or only one instance available")
        
        # Set empty results to indicate statistics was skipped
        sb.stat_results = None
        print("Statistics analysis skipped")
        
    else:
        # Multiple instances - run full statistics
        print("\nMultiple instances detected - running statistics analysis")
        
        # Set up statistics parameters
        train_test_split = 0.8
        metrics = ['Key=Response', 'Key=PerfRatio', 'Key=SuccProb', 'Key=MeanTime'] # RTT, 'InvPerfRatio'
        stParams = stats.StatsParameters(metrics=metrics, stats_measures=[stats.Median()])
        
        # Disable split validity check to avoid warnings
        original_check = getattr(training, 'check_split_validity', True)
        training.check_split_validity = False
        
        try:
            sb.run_Stats(stParams, train_test_split)
            print("Statistics analysis completed")
            
            if hasattr(sb, 'stat_results') and sb.stat_results is not None:
                print(f"Statistics results shape: {sb.stat_results.shape}")
                print("Sample statistics results:")
                display(sb.stat_results.head())
            else:
                print("No statistics results available")
                
        except Exception as e:
            print(f"Statistics analysis failed: {e}")
            import traceback
            traceback.print_exc()
        finally:
            # Restore original setting
            training.check_split_validity = original_check
            
else:
    print("No interpolation or bootstrap fallback results available - cannot run statistics analysis")


In [ ]:
# Check if we should run baseline analysis for single instance
if hasattr(sb, 'interp_results') and sb.interp_results is not None:
    n_rows = len(sb.interp_results)
    print(f"Data available for baseline analysis: {n_rows} rows")
    
    # Check if this is single instance by looking at original data
    is_single_instance = len(all_qaoa_df) == 1 if 'all_qaoa_df' in locals() else n_rows <= 5
    
    if is_single_instance:
        print("Single instance detected - baseline analysis provides limited value")
        print("Skipping baseline analysis for single instance case")
        print("Reason: Baseline comparison requires multiple instances or parameter configurations")
        
        # Set baseline to None to indicate it was appropriately skipped
        sb.baseline = None
        print("Baseline analysis skipped")
        
    else:
        # Multiple instances - run baseline analysis
        print("Multiple instances detected - running baseline analysis")
        
        # Check if train column exists, add if missing
        if 'train' not in sb.interp_results.columns:
            print("Adding train/test split column for baseline analysis...")
            np.random.seed(42)
            train_mask = np.random.random(len(sb.interp_results)) < 0.8
            sb.interp_results['train'] = train_mask.astype(int)
        
        try:
            sb.run_baseline()
            print("Baseline analysis completed")
            
            # Evaluate baseline
            if hasattr(sb, 'baseline') and sb.baseline is not None:
                recipes, _ = sb.baseline.evaluate()
                print(f"Baseline recipes shape: {recipes.shape}")
                print("\nBaseline recipe sample:")
                display(recipes.head())
            else:
                print("No baseline results available")
                
        except Exception as e:
            print(f"Baseline analysis failed: {e}")
            print("This may be due to insufficient data for meaningful baseline comparison")
            import traceback
            traceback.print_exc()
            
else:
    print("No interpolation results available - cannot run baseline analysis")

In [ ]:
# === Final Analysis Summary and Status Report ===
print("\n=== QAOA Stochastic Benchmark Analysis Summary ===")

# Original data visualization
if 'agg_df' in locals():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # PerfRatio vs training time
    valid_data = agg_df[agg_df['Approximation_Ratio'] != -999]
    if len(valid_data) > 0:
        ax1.scatter(valid_data['MeanTime'], valid_data['Approximation_Ratio'], alpha=0.7, color='blue')
        ax1.set_xlabel('Training Time (s)')
        ax1.set_ylabel('Approximation Ratio')
        ax1.set_title('QAOA Approximation Ratio vs Training Time')
        ax1.grid(True, alpha=0.3)
    
    # Parameter space visualization
    if 'p' in agg_df.columns and 'Approximation_Ratio' in agg_df.columns and len(valid_data) > 0:
        scatter = ax2.scatter(valid_data['p'], valid_data['Approximation_Ratio'],
                             c=valid_data['Approximation_Ratio'], alpha=0.7)
        ax2.set_xlabel('Circuit Depth')
        ax2.set_ylabel('Approximation Ratio')
        ax2.set_title('QAOA Parameter Space')
        ax2.grid(True, alpha=0.3)
        # Plot number of instances on colorbar for each depth
        num_instances = valid_data['p'].value_counts()
        scatter = ax2.scatter(valid_data['p'], valid_data['Approximation_Ratio'],
                             c=num_instances[valid_data['p']].values, alpha=0.7)
        plt.colorbar(scatter, ax=ax2, label='Number of Instances')
    plt.tight_layout()
    plt.show()
    
    # Data summary
    print(f"\nOriginal Data Summary:")
    print(f"Processed {len(all_qaoa_results)} QAOA optimization trials")
    if len(valid_data) > 0:
        print(f"Parameter space: Gamma [{valid_data['gamma'].min():.3f}, {valid_data['gamma'].max():.3f}], "
              f"Beta [{valid_data['beta'].min():.3f}, {valid_data['beta'].max():.3f}]", f"p: {valid_data['p'].unique()}")
        print(f"Approximation Ratio range: [{valid_data['Approximation_Ratio'].min():.4f}, {valid_data['Approximation_Ratio'].max():.4f}]")
        print(f"Training time range: [{valid_data['MeanTime'].min():.4f}, {valid_data['MeanTime'].max():.4f}] seconds")

# === Analysis Components Status ===
print(f"\n=== Analysis Components Status ===")
completed_components = []
skipped_components = []
failed_components = []

# Bootstrap
if hasattr(sb, 'bs_results') and sb.bs_results is not None:
    completed_components.append('Bootstrap')
    print(f"Bootstrap: COMPLETED ({sb.bs_results.shape[0]} results)")
else:
    failed_components.append('Bootstrap')
    print("Bootstrap: FAILED")

# Interpolation
if hasattr(sb, 'interp_results') and sb.interp_results is not None:
    completed_components.append('Interpolation')
    print(f"Interpolation: COMPLETED ({sb.interp_results.shape[0]} results)")
else:
    failed_components.append('Interpolation')
    print("Interpolation: FAILED")

# Statistics
if hasattr(sb, 'stat_results'):
    if sb.stat_results is not None:
        completed_components.append('Statistics')
        print(f"Statistics: COMPLETED ({sb.stat_results.shape[0]} results)")
    else:
        skipped_components.append('Statistics')
        print("Statistics: SKIPPED (single instance or insufficient data)")
else:
    failed_components.append('Statistics')
    print("Statistics: FAILED")

# Baseline
if hasattr(sb, 'baseline') and sb.baseline is not None:
    completed_components.append('Baseline')
    print("Baseline: COMPLETED")
else:
    # Check if baseline was skipped for single instance
    if ('qaoa_results' in locals() and len(qaoa_results) == 1) or \
       (hasattr(sb, 'interp_results') and len(sb.interp_results) <= 5):
        skipped_components.append('Baseline')
        print("Baseline: SKIPPED (single instance, limited baseline value)")
    else:
        failed_components.append('Baseline')
        print("Baseline: FAILED")

# Overall status
print(f"\n=== Overall Status ===")
total_components = 4
actual_completed = len(completed_components)
actual_attempted = total_components - len(skipped_components)

print(f"Completed components: {', '.join(completed_components) if completed_components else 'None'}")
if skipped_components:
    print(f"Skipped components: {', '.join(skipped_components)} (appropriate for single instance)")
if failed_components:
    print(f"Failed components: {', '.join(failed_components)}")

# Adjusted completion rate
if actual_attempted > 0:
    completion_rate = actual_completed / actual_attempted * 100
    print(f"Analysis completion rate: {completion_rate:.0f}% ({actual_completed}/{actual_attempted} applicable components)")
else:
    print("No applicable components for analysis")

# Results summary
if completed_components:
    print(f"\nCompleted analysis components: {len(completed_components)}")
else:
    print("\nNo analysis components completed successfully")
